# EDA — APTOS 2019 Blindness Detection

Análisis exploratorio del dataset. Responde cuatro preguntas que deciden el resto del proyecto:

1. **¿Qué tan desbalanceadas están las clases?** → pesos por clase en `train.py`.
2. **¿Las fotos vienen de cámaras distintas, y eso se relaciona con la clase?** → riesgo de que el modelo aprenda la cámara y no la enfermedad.
3. **¿Hay imágenes duplicadas, y con etiquetas distintas?** → `make_splits.py` las quita antes de dividir.
4. **¿Cómo se ven las fotos borrosas y oscuras?** → umbrales de `quality_filter.py`.

**Cómo correrlo (SageMaker Studio):** abrir con el kernel de Python 3 y *Run All*. La primera vez descarga el
dataset completo (~8 GB) a `DATA_DIR` y calcula estadísticas de las 3,662 imágenes (unos minutos); ambos
resultados se guardan y las siguientes veces carga rápido. Para una prueba rápida, poner `LIMIT = 64`.

> Antes de hacer commit: *Kernel → Restart Kernel and Clear Outputs* o al menos borrar las salidas de imágenes grandes.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Para importar src/ desde notebooks/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass

from src.common.preprocess import BACKGROUND_THRESHOLD, crop_retina, load_image
from src.data.download_kaggle import download_aptos

DATA_DIR = Path(os.environ.get("DATA_DIR", REPO / "data" / "aptos")).expanduser()
LIMIT = None          # None = todas las imágenes; un número (p. ej. 64) para probar rápido
CLASS_NAMES = {0: "Sin retinopatía", 1: "Leve", 2: "Moderada", 3: "Severa", 4: "Proliferativa"}
plt.rcParams["figure.dpi"] = 90
print("DATA_DIR =", DATA_DIR)

In [ ]:
download_aptos(DATA_DIR, limit=LIMIT)

df = pd.read_csv(DATA_DIR / "train.csv")
available = {p.stem for p in (DATA_DIR / "train_images").glob("*.png")}
df = df[df.id_code.isin(available)].reset_index(drop=True)
df["class_name"] = df.diagnosis.map(CLASS_NAMES)
print(f"{len(df)} imágenes con etiqueta")
df.head()

## 1. Distribución de clases

Para tamizaje nos importa la división **referible** (clase ≥ 2, moderada o más) contra **no referible**.

In [ ]:
counts = df.diagnosis.value_counts().sort_index()
summary = pd.DataFrame({
    "clase": [CLASS_NAMES[c] for c in counts.index],
    "imágenes": counts.values,
    "%": (100 * counts / counts.sum()).round(1).values,
}, index=counts.index)
display(summary)

referable = (df.diagnosis >= 2).mean()
print(f"Referibles (clase ≥ 2): {100 * referable:.1f}%  ·  No referibles: {100 * (1 - referable):.1f}%")
print(f"Desbalance: la clase más grande tiene {counts.max() / counts.min():.1f}× más imágenes que la más chica")

# Pesos por clase sugeridos para train.py (inverso de la frecuencia, normalizados a promedio 1)
weights = (1 / counts) / (1 / counts).mean()
print("Pesos por clase sugeridos:", {int(k): round(float(v), 2) for k, v in weights.items()})

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#2e7d32", "#2e7d32", "#f9a825", "#c62828", "#c62828"]  # colores del semáforo
ax.bar(summary.clase, summary["imágenes"], color=colors)
for i, v in enumerate(summary["imágenes"]):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_ylabel("Imágenes")
ax.set_title("Imágenes por clase (color = semáforo)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 2. Estadísticas por imagen

Una sola pasada sobre todas las imágenes calcula:

- **Resolución original** (alto × ancho): sirve como pista de qué cámara tomó la foto.
- **Hash perceptual** (`phash` de `imagehash`, 256 bits) de la retina recortada: dos fotos casi iguales tienen hashes casi iguales.
- **Nitidez**: varianza del Laplaciano sobre la retina recortada y llevada a 512×512 (valores bajos = borrosa).
- **Brillo**: promedio del gris solo dentro de la retina, sin contar el fondo negro (valores bajos = oscura).

Se guarda en `DATA_DIR/image_stats.csv` para no recalcular.

In [ ]:
QUALITY_SIZE = 512  # tamaño fijo para que la nitidez sea comparable entre cámaras


def blur_and_brightness(img_rgb):
    """Misma medición que usará quality_filter.py (sobre la retina ya recortada)."""
    gray = cv2.cvtColor(cv2.resize(img_rgb, (QUALITY_SIZE, QUALITY_SIZE), interpolation=cv2.INTER_AREA),
                        cv2.COLOR_RGB2GRAY)
    retina = gray > BACKGROUND_THRESHOLD
    blur = float(cv2.Laplacian(gray, cv2.CV_64F)[retina].var()) if retina.any() else 0.0
    brightness = float(gray[retina].mean()) if retina.any() else 0.0
    return blur, brightness


def image_stats(image_id):
    img = load_image(DATA_DIR / "train_images" / f"{image_id}.png")
    h, w = img.shape[:2]
    cropped = crop_retina(img)
    blur, brightness = blur_and_brightness(cropped)
    small_gray = cv2.cvtColor(cv2.resize(cropped, (256, 256), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2GRAY)
    phash = str(imagehash.phash(Image.fromarray(small_gray), hash_size=16))  # 256 bits
    return {"id_code": image_id, "height": h, "width": w, "phash": phash,
            "blur": blur, "brightness": brightness}


stats_path = DATA_DIR / "image_stats.csv"
cached = pd.read_csv(stats_path, dtype={"phash": str}) if stats_path.exists() else pd.DataFrame(columns=["id_code"])
missing = sorted(set(df.id_code) - set(cached.id_code))
if missing:
    print(f"Calculando estadísticas de {len(missing)} imágenes...")
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        new = pd.DataFrame(pool.map(image_stats, missing))
    cached = pd.concat([cached, new], ignore_index=True)
    cached.to_csv(stats_path, index=False)

cached[["height", "width"]] = cached[["height", "width"]].astype(int)
stats = df.merge(cached, on="id_code")
stats["resolution"] = stats.height.astype(str) + "×" + stats.width.astype(str)
stats[["height", "width", "blur", "brightness"]].describe().round(1)

## 3. Resoluciones (cámaras) y su relación con la clase

Si una resolución aparece casi solo en una clase, el modelo podría aprender a reconocer la cámara en
lugar de la retinopatía (*shortcut learning*). El recorte y el cambio a 384×384 de `preprocess.py`
reducen esa pista, pero conviene saberlo y vigilarlo en la matriz de confusión.

In [ ]:
top = stats.resolution.value_counts()
print(f"{len(top)} resoluciones distintas; las 10 más comunes cubren {100 * top.head(10).sum() / len(stats):.0f}% de las fotos")

common = top.head(10).index
table = pd.crosstab(stats.resolution[stats.resolution.isin(common)], stats.diagnosis[stats.resolution.isin(common)],
                    normalize="index").mul(100).round(0)
table.columns = [CLASS_NAMES[c] for c in table.columns]
table.insert(0, "imágenes", top[common])
display(table.sort_values("imágenes", ascending=False))

## 4. Ejemplos por clase y recorte de la retina

Arriba la foto original, abajo lo que produce `crop_retina()` (sin borde negro y cuadrada). Revisar que
el recorte funcione con las distintas cámaras.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for col, cls in enumerate(sorted(CLASS_NAMES)):
    ids = stats.id_code[stats.diagnosis == cls].to_numpy()
    if len(ids) == 0:
        axes[0, col].axis("off"); axes[1, col].axis("off"); continue
    img = load_image(DATA_DIR / "train_images" / f"{rng.choice(ids)}.png")
    axes[0, col].imshow(img); axes[0, col].set_title(f"{cls}: {CLASS_NAMES[cls]}\n{img.shape[0]}×{img.shape[1]}")
    axes[1, col].imshow(crop_retina(img)); axes[1, col].set_title("recortada")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Duplicados

Dos imágenes son casi duplicadas si la distancia de Hamming entre sus `phash` es ≤ `HASH_DISTANCE`
(de 256 bits). Importa por dos razones:

- Si una copia queda en entrenamiento y otra en prueba, las métricas salen infladas.
- Si las copias tienen **etiquetas distintas**, no sabemos cuál es la correcta: hay que quitarlas.

Los grupos se guardan en `DATA_DIR/duplicates.csv` para que `make_splits.py` los use.

In [ ]:
# Calibrado con una muestra: copias de la misma foto (otro tamaño, brillo, JPEG) quedan a <= 26;
# ojos distintos, a >= 32. El hash de 64 bits por defecto NO sirve: confunde ojos distintos que
# solo se parecen en la composición (disco óptico del mismo lado, mismo tono).
HASH_DISTANCE = 20

bits = np.array([imagehash.hex_to_hash(h).hash.flatten() for h in stats.phash], dtype=np.float32)
# Distancia de Hamming entre todos los pares, con multiplicación de matrices
dist = bits @ (1 - bits).T + (1 - bits) @ bits.T
i, j = np.where(np.triu(dist <= HASH_DISTANCE, k=1))
print(f"{len(i)} pares de imágenes casi iguales (distancia ≤ {HASH_DISTANCE})")

# Agrupar pares en grupos (componentes conexas)
parent = list(range(len(stats)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(i, j):
    parent[find(a)] = find(b)

stats["dup_group"] = [find(k) for k in range(len(stats))]
group_size = stats.dup_group.map(stats.dup_group.value_counts())
dups = stats[group_size > 1].copy()
n_labels = dups.groupby("dup_group").diagnosis.transform("nunique")
dups["conflicting_labels"] = n_labels > 1

n_groups = dups.dup_group.nunique()
n_conflict = dups[dups.conflicting_labels].dup_group.nunique()
print(f"{n_groups} grupos de duplicados con {len(dups)} imágenes en total")
print(f"{n_conflict} grupos tienen etiquetas distintas ({dups.conflicting_labels.sum()} imágenes)")
removed = len(dups) - (n_groups - n_conflict)  # se queda una por grupo sin conflicto
print(f"Si se deja una imagen por grupo sin conflicto y se quitan los grupos con conflicto, "
      f"quedan {len(stats) - removed} de {len(stats)} imágenes")

dups[["id_code", "diagnosis", "dup_group", "conflicting_labels"]].sort_values("dup_group").to_csv(
    DATA_DIR / "duplicates.csv", index=False)

In [ ]:
# Revisar a ojo: primero los grupos con etiquetas distintas, luego los demás
order = dups.sort_values("conflicting_labels", ascending=False).dup_group.unique()
show = order[:6]
if len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
    for row, g in enumerate(show):
        members = dups[dups.dup_group == g].head(2)
        for col, (_, r) in enumerate(members.iterrows()):
            axes[row, col].imshow(crop_retina(load_image(DATA_DIR / "train_images" / f"{r.id_code}.png")))
            axes[row, col].set_title(f"{r.id_code}\nclase {r.diagnosis}" + ("  (conflicto)" if r.conflicting_labels else ""))
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No hay duplicados")

## 6. Calidad: fotos borrosas y oscuras

El dataset tiene pocas fotos realmente malas, así que proponemos umbrales en los percentiles bajos
(`QUALITY_PERCENTILE`) y luego **los revisamos a ojo** con los ejemplos de abajo. Los valores finales
se copian como constantes en `src/common/quality_filter.py`.

Ojo: la varianza del Laplaciano también baja en fotos oscuras aunque estén enfocadas (hay menos
contraste), así que las dos mediciones se traslapan. Por eso hay que ver los ejemplos antes de fijar los umbrales.

In [ ]:
QUALITY_PERCENTILE = 2  # % de fotos que el filtro rechazaría en este dataset

blur_threshold = float(np.percentile(stats.blur, QUALITY_PERCENTILE))
dark_threshold = float(np.percentile(stats.brightness, QUALITY_PERCENTILE))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(np.log10(stats.blur + 1), bins=60, color="#2e75b6")
axes[0].axvline(np.log10(blur_threshold + 1), color="#c62828", ls="--", label=f"umbral = {blur_threshold:.1f}")
axes[0].set_title("Nitidez (log10 de la varianza del Laplaciano)"); axes[0].legend()
axes[1].hist(stats.brightness, bins=60, color="#2e75b6")
axes[1].axvline(dark_threshold, color="#c62828", ls="--", label=f"umbral = {dark_threshold:.1f}")
axes[1].set_title("Brillo dentro de la retina (0-255)"); axes[1].legend()
plt.tight_layout()
plt.show()

rejected = (stats.blur < blur_threshold) | (stats.brightness < dark_threshold)
print(f"Con estos umbrales se rechazaría {100 * rejected.mean():.1f}% de las fotos")
print("Rechazo por clase (%):", (stats[rejected].diagnosis.value_counts(normalize=True).sort_index() * 100).round(0).to_dict())

In [ ]:
def show_extremes(column, title, n=6):
    worst = stats.nsmallest(n, column)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3))
    for ax, (_, r) in zip(np.atleast_1d(axes), worst.iterrows()):
        ax.imshow(crop_retina(load_image(DATA_DIR / "train_images" / f"{r.id_code}.png")))
        ax.set_title(f"{column}={r[column]:.1f}\nclase {r.diagnosis}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_extremes("blur", "Las más borrosas")
show_extremes("brightness", "Las más oscuras")

In [ ]:
# Copiar estas constantes a src/common/quality_filter.py (ajustarlas si los ejemplos de arriba no convencen)
print(f"QUALITY_SIZE = {QUALITY_SIZE}")
print(f"BLUR_THRESHOLD = {blur_threshold:.1f}        # varianza del Laplaciano; menor = borrosa")
print(f"BRIGHTNESS_THRESHOLD = {dark_threshold:.1f}   # gris promedio de la retina; menor = oscura")

## 7. Conclusiones

Llenar después de correr el notebook completo:

- **Desbalance:** …
- **Cámaras / resoluciones:** …
- **Duplicados:** … grupos, … con etiquetas distintas → `make_splits.py` deja una imagen por grupo y quita los grupos con conflicto.
- **Calidad:** umbrales `BLUR_THRESHOLD = …`, `BRIGHTNESS_THRESHOLD = …` (rechazan …% del dataset).